# ⚠️ 重要：DataBricks と NumPy の互換性に関する注意事項

> **DataBricks と NumPy の相性は最悪に近い（Repeat!!!）**

## 発生する問題

| 問題 | 詳細 |
|------|------|
| **Kernel Crash** | NumPy の大規模配列操作時にカーネルがクラッシュしやすい |
| **回避困難** | 根本的な解決策がなく、対症療法的な対策が必要 |

## 対策：こまめなメモリ管理

このノートブックでは以下の **メモリ管理戦略** を一貫して採用している：

```python
sdf_tmp.unpersist()         # Spark DataFrame のキャッシュを解放
del sdf_tmp, np_tmp         # 不要になった変数を即座に削除
spark.catalog.clearCache()  # Spark の全キャッシュをクリア
gc.collect()                # Python GC を強制実行してメモリを回収
```

> **運用ルール：** 大きな行列・DataFrame の処理が完了したら、**必ず直後にメモリ解放処理を実行すること。**

In [0]:
%pip install                                                         \
        numpy                 sentencepiece       protobuf           \
        adlfs                 fsspec              azure-storage-blob \
        readability-lxml      w3lib               httpx              \
        beautifulsoup4        azure-ai-inference  packaging          \
        python-dotenv         openai              polars             \
        sentence-transformers tiktoken            fastembed

%restart_python

## 📦 ライブラリのインポート

| カテゴリ | ライブラリ | 用途 |
|----------|-----------|------|
| **システム** | `os`, `gc`, `ast`, `json`, `asyncio` | 環境変数取得・GC 強制実行・型安全なパース・JSON 操作・非同期制御 |
| **HTTP** | `httpx` | 非同期 HTTP クライアント（接続プール・タイムアウト管理） |
| **数値計算** | `numpy`, `scipy` | ベクトル演算・疎行列 (CSR) の生成と操作 |
| **データ処理** | `polars`, `pandas` | 高速 DataFrame 処理 / Spark 連携用 DataFrame |
| **可視化** | `matplotlib` | スコア分布のヒストグラム生成 |
| **分散処理** | `pyspark`, `delta` | Delta Lake 対応の大規模分散処理 |
| **LLM** | `openai`, `azure.ai.inference` | Azure AI Foundry 経由での LLM 呼び出し |
| **Embedding** | `sentence_transformers` | テキストの埋め込みベクトル生成（BAAI/bge-m3） |
| **自作モジュール** | `llm_agent.LlmAgent` | LLM 呼び出しの抽象化ラッパー（並列制御・リトライ込み） |

In [0]:
import os
import gc
import ast
import json
import asyncio
import httpx
import numpy  as np
import scipy  as sp
from dotenv       import load_dotenv
from openai       import AsyncOpenAI

import polars as pl
import matplotlib.pyplot as plt

import pandas as pd
from pyspark.sql               import SparkSession
from pyspark.sql.functions     import col
from delta                     import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage
from sentence_transformers     import SentenceTransformer

from llm_agent                 import LlmAgent

## ⚙️ PySpark 初期設定

SparkSession を Delta Lake 対応の設定で初期化する。

### 設定グループ一覧

| グループ | 主な設定キー | 目的 |
|----------|-------------|------|
| **コミットプロトコル** | `spark.sql.sources.commitProtocolClass` | Delta Lake 用 SQL コミットプロトコルを使用 |
| **AQE（適応的クエリ実行）** | `spark.sql.adaptive.enabled` | クエリ実行計画をランタイムで最適化 |
| **パーティション自動調整** | `spark.sql.shuffle.partitions` | シャッフル後のパーティション数を `auto` で動的制御 |
| **小ファイル統合** | `spark.sql.adaptive.coalescePartitions.enabled` | 過剰に分割されたパーティションを自動統合 |
| **ブロードキャスト結合** | `spark.sql.autoBroadcastJoinThreshold` | 10MB 以下のテーブルをブロードキャスト結合に自動変換 |
| **Arrow 最適化** | `spark.sql.execution.arrow.pyspark.enabled` | Pandas ↔ Spark 変換を Apache Arrow で高速化 |
| **Delta Lake 拡張** | `spark.sql.extensions` | Delta Lake 固有の SQL 構文・解析機能を有効化 |
| **Delta 書き込み最適化** | `spark.databricks.delta.optimizeWrite.enabled` | 書き込み時にシャッフルして大きなファイルを生成 |
| **Delta 自動最適化** | `spark.databricks.delta.autoCompact.enabled` | 書き込み後に小ファイルを自動統合 |
| **分離レベル** | `spark.databricks.delta.write.isolationLevel` | スナップショット分離でトランザクション整合性を保証 |

### リソース設定（クラスター依存）

```
spark.driver.memory        = 64g   ← クラスターサイズに応じて変更すること
spark.driver.maxResultSize = 32g
spark.rpc.message.maxSize  = 1024
```

> **Note:** ドライバーメモリ等のリソース設定は **クラスターサイズによって動的に書き換えること。**

In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # JavaSerializerからKryoSerializerへの変更
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Apache Arrow 形式で Pandas<=>Spark 変換を行う
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化

# 追加の設定処理
# クラスターサイズによって、動的に書き換えること
builder = builder\
			.config("spark.driver.memory", "64g")\
    		.config("spark.driver.maxResultSize", "32g")\
			.config("spark.rpc.message.maxSize", "1024")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# メモリ管理
del builder
gc.collect()


## 🔑 環境変数の読み込みと型変換

`.env` ファイルおよび Databricks ウィジェットからパラメータを取得し、適切な型に変換する。

### 環境変数（`.env` ファイル）

| 変数名 | 型 | 説明 |
|--------|----|------|
| `AI_FOUNDRY_ENDPOINT` | `str` | Azure AI Foundry のエンドポイント URL |
| `AI_FOUNDRY_API_KEY` | `str` | API 認証キー（⚠️ **絶対にログ出力しないこと**） |
| `AI_FOUNDRY_MODEL` | `str` | 使用する LLM モデル名 |
| `LLM_MAX_TOKENS` | `int` ← `str` | 生成トークンの上限数 |
| `LLM_TEMPERATURE` | `float` ← `str` | 生成の多様性（0.0 = 決定論的 / 1.0 = ランダム） |
| `LLM_TOP_P` | `float` ← `str` | 核サンプリングの閾値 |

### Databricks ウィジェット

| 変数名 | 型 | 説明 |
|--------|----|------|
| `BRAND_NAME` | `str` | 分析対象のブランド名 |
| `TARGET_BRAND_WORDS` | `list[str]` ← `str` | ブランドキーワードリスト（文字列または Python リスト形式の文字列） |
| `EXTRACTION_ID` | `int` ← `str` | 地点抽出バッチの識別 ID |

### ⚠️ 型変換の注意点

ウィジェット・環境変数はすべて **`str` 型** で返ってくるため、使用前に明示的な型変換が必要。

`TARGET_BRAND_WORDS` は以下いずれかの文字列フォーマットが入力される：

| 入力例 | 解釈後 |
|--------|--------|
| `"['aaa', 'bbb', 'ccc']"` | `['aaa', 'bbb', 'ccc']` |
| `"aaa"` or `"'aaa'"` | `['aaa']` |

`ast.literal_eval()` で安全にパースし、単一文字列の場合はリストに包む。

In [0]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")

# ウェジットからの設定の読み込み
BRAND_NAME                = dbutils.widgets.get('BRAND_NAME')
TARGET_BRAND_WORDS        = dbutils.widgets.get('TARGET_BRAND_WORDS')
EXTRACTION_ID             = dbutils.widgets.get('EXTRACTION_ID')

# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS            = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE           = float(LLM_TEMPERATURE)
LLM_TOP_P                 = float(LLM_TOP_P)
EXTRACTION_ID             = int(EXTRACTION_ID)

# メモ：
# TARGET_BRAND_WORDS の中身として
# "['aaa', 'bbb', 'ccc']"
# "aaa" or "'aaaaa'"
# 上記のような「文字列」を想定している
# 
# このような文字列を正しくパースするために、astライブラリを利用することとした
try:
	TARGET_BRAND_WORDS = ast.literal_eval(TARGET_BRAND_WORDS)
	if isinstance(TARGET_BRAND_WORDS, str):
		TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]
except (ValueError, SyntaxError):
	TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')
print(f'BRAND_NAME:                {BRAND_NAME}')
print(f'TARGET_BRAND_WORDS:        {TARGET_BRAND_WORDS}')
print(f'EXTRACTION_ID:             {EXTRACTION_ID}')

## 🔌 クライアント初期設定

LLM への非同期アクセスに必要なクライアント群を初期化する。

### コンポーネント構成

```
LlmAgent (llmClient)
├── AsyncOpenAI (openai_cli)
│   ├── base_url    : AI_FOUNDRY_ENDPOINT
│   ├── api_key     : AI_FOUNDRY_API_KEY
│   ├── max_retries : 3
│   └── httpx.AsyncClient (http_client)
│       ├── Limits  : keepalive=20, max_connections=300
│       └── Timeout : total=300s, connect=5s
├── model       : AI_FOUNDRY_MODEL
├── max_tokens  : LLM_MAX_TOKENS
├── temperature : LLM_TEMPERATURE
├── top_p       : LLM_TOP_P
└── asyncio.Semaphore (semaphore)  ← 同時実行数: 50
```

### パラメータ詳細

| 設定項目 | 値 | 説明 |
|----------|-----|------|
| `max_keepalive_connections` | 20 | TCP Keep-Alive 接続の最大数 |
| `max_connections` | 300 | HTTP 接続プール全体の上限 |
| `timeout (total)` | 300s | LLM 応答待ちの最大待機時間 |
| `timeout (connect)` | 5s | 接続確立のタイムアウト |
| `max_retries` | 3 | API エラー時の自動リトライ回数 |
| `semaphore` | 50 | 最大同時並列リクエスト数（スロットリング） |

In [0]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
semaphore   = asyncio.Semaphore(50)
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P), semaphore)



## 🔍 対象商品分析部

ブランドのキーワードリスト（`TARGET_BRAND_WORDS`）を LLM で分析し、商品の特徴・利用シーンを構造化する。  
結果はファイルにキャッシュされ、再実行時の API コストを削減する。

### 処理フロー

```
[開始]
  │
  ├─ キャッシュ確認: ブランドLP/{BRAND_NAME}.md が存在する？
  │
  ├─ 【No: 初回実行】
  │     1. `ブランドLP/商品分析.md` (システムプロンプト) を読み込む
  │     2. 【WORD_LIST】→ TARGET_BRAND_WORDS に置換
  │     3. LLM に商品分析を依頼（SystemMessage のみ）
  │     4. 結果を `ブランドLP/{BRAND_NAME}.md` に JSON 保存（キャッシュ）
  │
  └─ 【Yes: 2回目以降】
        キャッシュファイルから分析結果を読み込む（API 呼び出しをスキップ）

[出力] product_analysis: LLM による商品分析 JSON 文字列
```

> **Note:** キャッシュを削除すれば再分析が実行される。意図的に再分析したい場合は `ブランドLP/{BRAND_NAME}.md` を削除すること。

In [ ]:
output_file_path = f'ブランドLP/{BRAND_NAME}.md'

# 商品分析情報が存在しない場合
if not os.path.exists(output_file_path):
	with open(f'ブランドLP/商品分析.md', 'r', encoding='utf-8') as f:
		sys_prompt = f.read()
		sys_prompt = sys_prompt.replace('【WORD_LIST】', ', '.join(TARGET_BRAND_WORDS))

	messages  = []
	messages.append(SystemMessage(content=sys_prompt))
	product_analysis = await llmClient.complete(messages)
	with open(output_file_path, 'w', encoding='utf-8') as f:
		f.write(json.dumps(product_analysis, indent=4, ensure_ascii=False))

# 商品分析情報の読み込み
with open(output_file_path, 'r', encoding='utf-8') as f:
    product_analysis = f.read()
product_analysis

## 🤖 LLM によるベクトル生成部：Micro-Context キーワードの抽出

商品分析結果を LLM に渡し、「商品が輝くシーン（positive）」と「商品がミスマッチなシーン（negative）」を  
**重み付きキーワード（Micro-Context）** として抽出する。  
このキーワード群が、後段の Embedding → スポット係数計算の入力になる。

---

### 処理フロー

```
[product_analysis]  ← 商品分析 JSON（前セルで取得済み）
       │
       ▼
  SystemMessage（マーケティング専門家ロールのプロンプト）
  UserMessage  （product_analysis の内容）
       │
       ▼  LLM 呼び出し（llmClient.complete）
       │
       ▼
[analysis_data]  ← positive / negative キーワード × 重み の JSON
```

---

### INPUT

| 項目 | 内容 |
|------|------|
| **SystemMessage** | マーケティング専門家として Micro-Context を抽出するよう指示するプロンプト |
| **UserMessage** | `product_analysis`（前セルで取得した商品 LP の LLM 解析結果） |

---

### OUTPUT フォーマット

LLM には **生の JSON のみ** を返すよう指示する（Markdown コードブロック禁止）。  
各キーワードをキー、親和度スコア（0.0〜1.0）を値とする辞書形式。

```json
{
    "positive": {
        "コラボカフェの推し色テーブルで生誕祭の祭壇を即席セット": 0.95,
        "ライブ会場のグッズ交換エリアで缶バッジのトレード待機":   0.93,
        ...  （上位 20 個程度）
    },
    "negative": {
        "満員電車の車内で祭壇を広げて撮影しようとする":     0.98,
        "ライブ本編中の客席で自立台座を設置して視界を遮る": 0.97,
        ...  （上位 12 個程度）
    }
}
```

---

### キーワード設計の指針（プロンプトで LLM に指示している内容）

**Micro-Context とは「誰が・どこで・何をしているか」が具体的に想像できる複合キーワード。**  
単一の一般名詞（「公園」「オフィス」）は禁止し、必ず「場所＋状況」または「属性＋場所」の形式で抽出させる。

| 分類 | キーワードの種類 | NG 例 | OK 例 |
|------|----------------|-------|-------|
| **positive** | 商品が必須・魅力を最大化するシーン | 「ジム」「キャンプ」 | 「深夜の 24 時間ジム」「雨上がりのオートキャンプ場」 |
| **negative** | 商品の機能が死ぬ・邪魔になるシーン | 「電車」「図書館」 | 「満員電車の通勤客」「試験中の図書館自習室」 |

### スコアの意味

| スコア | 意味 |
|--------|------|
| **1.0** | 完全にその商品の独壇場（または絶対に使用不可） |
| **0.8** | 非常に相性が良い（または強い懸念がある） |
| **0.5** | 条件による（出力対象外） |

> **Note:** LLM の出力は文字列として返ってくるため、後続セルで `json.loads()` によるパースが必要。

In [0]:
messages  = []
messages.append(SystemMessage(content=(
				"あなたは提供された「商品分析結果」を基に、その商品の利用文脈（Micro-Context）を特定するマーケティングの専門家です。\n"
				"提供された商品情報を分析し、その商品が「最高に輝く具体的なシーン（適合）」と「全く役に立たない、あるいはミスマッチなシーン（不適合）」を洗い出してください。\n\n"
				
				"【重要な指示】\n"
                "出力するキーワードは、ベクトル検索のクエリとして使用されます。\n"
				"そのため、単一の一般名詞（例：「公園」「オフィス」）は **禁止** です。\n"
                "必ず **「場所＋状況」** または **「属性＋場所」** の複合キーワード（Micro-Context）を選定してください。\n"
    			"抽出する単語は、単なる一般名詞ではなく、「誰が・どこで・何をしているか」がありありと想像できるような、具体的かつ解像度の高いキーワードを選定してください。\n"
    			"特に「場所」に関しては、大分類（例：公園）ではなく、詳細な施設タイプ（例：ドッグラン、親水広場）や、利用目的が明確なスポット名を優先してください。\n"
                "LPに直接記載がなくても、商品の特性から論理的に推測されるシーンは積極的に広げて記述してください。\n"
                "・NG例: 「ジム」「キャンプ」「サラリーマン」\n"
                "・OK例: 「深夜の24時間ジム」「雨上がりのオートキャンプ場」「満員電車の通勤客」「コンセントのあるカフェ席」\n\n"

				"【出力形式（厳守）】\n"
				"回答は必ず以下のJSON形式のみを出力してください。\n"
				"各単語をキーとし、その関連度の強さ（重み）を 0.0〜1.0 の数値で値として設定してください。\n"
				"Markdown記法（```json 等）は含めず、生のJSONテキストのみを返してください。\n"
                "・Positiveとして、確信度の高い上位20個程度を抽出してください。\n"
                "・Negativeとして、確信度の高い上位12個程度を抽出してください。\n\n"
					
				"""
				{
					"positive": {"複合キーワードA": 0.89, "キーワードB": 0.70, ...},
					"negative": {"複合キーワードC": 0.91, ...},
				}
				"""
				"\n\n"
				
				"【分析の視点】\n"
				"--- positive（適合）：商品が必須となる、または魅力を最大化する文脈 ---\n"
				"1. 具体的な施設・スポット（Places）：\n"
				"   - 抽象的な「店」「屋外」はNG。\n"
				"   - 「24時間ジム」「オートキャンプ場」「コワーキングスペース」など、行動が特定できる施設名。\n"
				"2. 利用シーン・瞬間（Scenes）：\n"
				"   - 「通勤ラッシュ」「運動後のシャワー」「子供の寝かしつけ」など、具体的なタイムラインや状況。\n"
				"3. ターゲットの属性・状態（Traits）：\n"
				"   - 「健康志向」のような広い言葉より、「糖質制限中」「リモートワーク疲れ」など具体的な状態。\n"
                "4. 物理的適合 (Physical Fit):\n"
                "   - 商品のサイズ、電源、耐久性が、その場所の設備・環境と完璧に噛み合うか。\n"
                "   - マグネットでくっつく防水Bluetoothスピーカー  →  「ユニットバスの壁面」「雨の日のキャンプのタープ下」\n"
				"5. 心理的・行動的適合 (Contextual Fit):\n"
                "   - その場所にいる人の「特定の悩み」を解決するか。\n"
				"   - 周囲の音を消すデジタル耳栓  →  「いびきが気になるカプセルホテル」「瞑想に集中したいヨガスタジオの隅」\n\n"
				
				"--- negative（不適合）：商品の機能が死ぬ、または邪魔になる文脈 ---\n"
				"1. 阻害要因となる場所（Places）：\n"
				"   - 商品のスペック（大きさ、音、電源有無など）的に使えない場所（例：図書館、満員電車）。\n"
				"2. 無意味なシーン（Scenes）：\n"
				"   - その商品をあえて使う必要がない状況。\n"
                "3. 環境不適合 (Environmental Mismatch):\n"
                "   - 商品を使うには狭すぎる、暗すぎる、うるさすぎる、静かすぎる場所。\n"
                "4. マナー・ルール違反 (Social Mismatch):\n"
                "   - その場所でその商品を使うことが「白い目」で見られる、あるいは禁止されている。\n\n"
				
				"【重み（スコア）の基準】\n"
				"・1.0に近いほど：その傾向が非常に強い、確信度が高い\n"
				"・0.0に近いほど：関連性が薄い\n"
                "・1.0: 完全にその商品の独壇場である（または絶対に使用不可である）。\n"
                "・0.8: 非常に相性が良い（または強い懸念がある）。\n"
                "・0.5: 条件による（今回は出力対象外）。\n"
				"・positiveの場合：適合度の高さ\n"
				"・negativeの場合：不適合度の高さ（明確に避けるべき度合い）\n\n"
			)))
messages.append(UserMessage(content=product_analysis))
analysis_data = await llmClient.complete(messages)
analysis_data

## 📍 ADIDリスト取得部

Spark を使用して、指定された地点バッチ（`EXTRACTION_ID`）に来訪した ADID（広告識別子）のリストを取得する。

### 対象テーブル

| テーブル名 | 説明 | 使用カラム |
|------------|------|------------|
| `envprod.polygonlist` | ポリゴン（地点）定義マスタ | `project`, `caption`, `extraction_id`, `branch_id`, `spot_id` |
| `envprod.source` | ADID × 地点の来訪ログ | `extraction_id`, `branch_id`, `spot_id`, `adid` |

### 処理フロー（JOIN 構造）

```
polygonlist                    source（来訪ログ）
  └── filter: extraction_id      └── filter: extraction_id
      == EXTRACTION_ID               == EXTRACTION_ID
                    │                         │
                    └──── INNER JOIN ──────────┘
                          on: extraction_id, branch_id, spot_id
                                    │
                              select: caption, adid
                              dropDuplicates
                                    │
                    ┌───────────────┴────────────────┐
                    ▼                                ▼
         target_adidlist                      pdf_adidlist
         (list[str])                          (pandas DataFrame)
         ユニーク ADID リスト               caption × adid の対応表
```

### 出力変数

| 変数名 | 型 | 説明 |
|--------|----|------|
| `target_adidlist` | `list[str]` | ユニーク ADID のリスト（後続の行列抽出に使用） |
| `pdf_adidlist` | `pd.DataFrame` | `caption`（地点名） × `adid` の対応表 |

> **Note:** Spark DataFrame は処理後に `.unpersist()` と `clearCache()` で即座に解放する。

In [ ]:
# 特定の地点に対するADIDリストを取得
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)
ADIDLIST_TABLE    = "adinte_adintedmp.envprod.source"
sdf_adidlist      = spark.read.table(ADIDLIST_TABLE)\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
                            .select(['caption', 'adid'])\
                            .dropDuplicates()

target_adidlist   = [row['adid']    for row in sdf_adidlist.select('adid').dropDuplicates().toLocalIterator()]
pdf_adidlist      = sdf_adidlist.toPandas()

# メモリ管理
sdf_polygonlist.unpersist()
sdf_adidlist.unpersist()
del sdf_polygonlist, sdf_adidlist
spark.catalog.clearCache()
gc.collect()

pdf_adidlist

## 📊 各種特徴ベクトル取得部

事前計算済みのコホート係数行列とキャプション文脈行列を読み込み、対象 ADID のベクトルを抽出・正規化する。

### 使用ファイル

| ファイル | 場所 | 形状 | 説明 |
|----------|------|------|------|
| `cohort.npz` | Azure Volume | `(全ADID数, コホート数)` | ADID × コホートキャプションの疎行列（CSR 形式） |
| `cohort_caption_matrix.npz` | ローカル | `(コホート数, スポット数)` | コホートキャプション → スポットへの変換行列 |

### `cohort.npz` の内部構造

| キー | 説明 |
|------|------|
| `data` | CSR 形式の非ゼロ値配列 |
| `indices` | 列インデックス配列 |
| `indptr` | 行ごとのスライスインデックス |
| `shape` | 行列のサイズ `(ADID数, コホート数)` |
| `adid_list` | ADID のリスト（行方向） |
| `business_codelist` | コホートキャプション ID のリスト（列方向） |

### 処理フロー

```
1. cohort.npz 読み込み
   └── CSR 疎行列として展開 → np_cohort

2. target_adidlist に含まれる ADID のみ行方向に抽出
   └── pd.Index.get_indexer() でインデックス変換

3. L2 正規化（各 ADID の行ベクトルの長さを 1 に統一）
   └── ※ 疎行列のため diags(1/norm) @ matrix で実現
   └── ※ 既に正規化済みの場合はスキップ（allclose チェック）

4. cohort_caption_matrix.npz 読み込み → spots_matrix

5. 整合性チェック（assert × 4）
   ├── 全行の L2 ノルム ≈ 1.0
   ├── コホートキャプション ID が一致
   ├── スポット一覧が一致
   └── np_cohort の列数 == spots_matrix の行数
```

### ⚠️ 疎行列に対する L2 正規化

密行列なら `sklearn.preprocessing.normalize()` で直接実行できるが、疎行列には使えない。  
そのため **疎な対角行列** を経由して実現している：

```python
l2norm_vector = scipy.sparse.linalg.norm(np_cohort, axis=1)
np_cohort     = scipy.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort
```

In [0]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]
np_codelist = npz["business_codelist"]

# 特定のADIDのみの抜き出し
indexer     = pd.Index(np_adidlist)
indices     = indexer.get_indexer(target_adidlist)
indices     = indices[indices != -1]
np_cohort   = np_cohort[indices, :]
np_adidlist = np_adidlist[indices]


# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort

	# メモリ管理
	del l2norm_vector

# メモリ管理
del npz, indexer, indices
gc.collect()


# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
spots_matrix           = npz["data"]
relational_codelist    = npz["business_codelist"]
relational_spots       = npz["business_placelist"]
dict_code2name         = npz["dict_code2name"].item()

# 簡易チェック
assert np.allclose(np_cohort.power(2).sum(axis=1), 1.0), "Not all row vectors have a unit length (squared norm != 1.0)."
assert np.array_equal(relational_codelist, np_codelist), "There are unregistered keys."
assert np.array_equal(np.array([[code, *dict_code2name[code]] for code in np_codelist]), relational_spots), "Location list is inconsistent."
assert spots_matrix.shape[0] == np_cohort.shape[1], "Dimension mismatch: spots_matrix rows (count: {}) do not align with np_cohort columns (count: {})" .format(spots_matrix.shape[0], np_cohort.shape[1])

# メモリ管理
del np_codelist
del npz
del relational_codelist, relational_spots, dict_code2name
gc.collect()

print(np_cohort.shape)
print(spots_matrix.shape)

## 🧮 計算部（本命）：LP ベクトルの生成

LLM が出力した「商品シーン × 重み」を、スポット空間上の係数ベクトルに変換する。  
この係数ベクトルが、ADID ごとの親和性スコア算出のコアになる。

---

### ステップ概要

```
[LLM 出力 JSON]
      │
      ▼ Step 1: 形式変換（positive/negative → 符号付きリスト）
[lp_keywords: list[str], lp_weights: list[float]]
      │
      ▼ Step 2: L1 正規化（重みの総和を 1 に統一）
[lp_weights: 正規化済み]
      │
      ▼ Step 3: Embedding モデル（BAAI/bge-m3）で高次元空間へ写像
[lp_matrix: M × 1024]
      │
      ▼ Step 4: 行列積でスポット係数ベクトルを算出
[lp_coefficient: 1 × スポット数S]
      │
      ▼ Step 5: L2 正規化（ベクトルの長さを 1 に統一）
[lp_coefficient: L2 正規化済み]
```

---

### Step 1｜LLM 出力形式の変換

LLM が返す JSON を `(keywords, weights)` の平行リストに変換する。  
`negative` シーンは **重みを反転（符号をマイナス）** することで、ベクトル空間上で「逆方向」を表現する。

**INPUT（LLM 出力 JSON）**
```json
{
    "positive": {
        "コラボカフェの推し色テーブルで生誕祭の祭壇を即席セット": 0.95,
        "ライブ会場のグッズ交換エリアで缶バッジのトレード待機":   0.93,
        ...
    },
    "negative": {
        "満員電車の車内で祭壇を広げて撮影しようとする":     0.98,
        "ライブ本編中の客席で自立台座を設置して視界を遮る": 0.97,
        ...
    }
}
```

**OUTPUT（平行リスト）**
```python
lp_keywords = ["コラボカフェの...", "ライブ会場の...", ..., "満員電車の...", "ライブ本編中の...", ...]
lp_weights  = [+0.95,              +0.93,             ..., -0.98,           -0.97,               ...]
#              ↑ positive はそのまま                        ↑ negative は符号反転
```

---

### Step 2｜L1 正規化

重みベクトルの絶対値の総和が 1 になるよう正規化する。  
positive / negative の比率にかかわらず、重みのスケールを統一するためのステップ。

$$\vec{w}_{L1} = \frac{\vec{w}}{\|\vec{w}\|_1} = \frac{\vec{w}}{\sum_i |w_i|}$$

---

### Step 3｜Embedding モデルによる高次元空間への写像

`BAAI/bge-m3` モデルで各キーワードを 1024 次元の埋め込みベクトルに変換する。  
`normalize_embeddings=True` により、各行ベクトルは L2 正規化済み（長さ = 1）。

```python
lp_vector = lp_weights.reshape(1, -1)                             # 形状: 1 × M
lp_matrix = model.encode(lp_keywords, normalize_embeddings=True)  # 形状: M × 1024
```

**`lp_matrix` のデータ構造イメージ（M × 1024）**

| 商品シーン（M 行）| $d_1$ | $d_2$ | $\cdots$ | $d_{1024}$ |
| :--- | :---: | :---: | :---: | :---: |
| コラボカフェの推し色テーブルで... | 0.012 | −0.045 | $\cdots$ | 0.088 |
| ライブ会場のグッズ交換エリアで... | −0.023 | 0.071 | $\cdots$ | −0.012 |
| $\vdots$ | $\vdots$ | $\vdots$ | $\ddots$ | $\vdots$ |
| 満員電車の車内で祭壇を広げて... | 0.055 | −0.003 | $\cdots$ | 0.041 |
| ライブ本編中の客席で自立台座を... | 0.099 | −0.065 | $\cdots$ | 0.010 |
| $\vdots$ | $\vdots$ | $\vdots$ | $\ddots$ | $\vdots$ |

---

### Step 4｜LP からみたスポットの重要度ベクトル

3 つの行列を連鎖的に掛け合わせ、「この商品がどのスポットと親和性が高いか」を表す **1 × S の係数ベクトル** を求める。

$$\underbrace{\vec{lp\_coefficient}}_{1 \times S} = \underbrace{\vec{lp\_vector}}_{1 \times M} \cdot \underbrace{lp\_matrix}_{M \times 1024} \cdot \underbrace{spots\_matrix^T}_{1024 \times S}$$

| 行列 | 形状 | 意味 |
|------|------|------|
| `lp_vector` | $1 \times M$ | L1 正規化済みの重みベクトル（正負付き） |
| `lp_matrix` | $M \times 1024$ | 各キーワードの Embedding（L2 正規化済み） |
| `spots_matrix.T` | $1024 \times S$ | 各スポットの Embedding（コホート空間へのブリッジ） |
| **`lp_coefficient`** | $1 \times S$ | **各スポットに対する LP の親和度スコア** |

> **直感的な理解：** LLM が「このシーンに合う」と判断したキーワードほど大きく寄与し、  
> そのキーワードと意味的に近いスポットのスコアが高くなる。逆に negative キーワードに近いスポットは低くなる。

---

### Step 5｜L2 正規化

`lp_coefficient` の方向（各スポットへの相対的な親和度の比率）はそのままに、  
ベクトルの長さを 1 に揃える。後続のコサイン類似度計算との整合性を保つため。

$$\vec{lp\_coefficient}_{L2} = \frac{\vec{lp\_coefficient}}{\|\vec{lp\_coefficient}\|_2}$$

In [0]:
# LLMの出力をベクトル化
items_positive   = list(analysis_data['positive'].items())
items_negative   = list(analysis_data['negative'].items())
lp_keywords      = [key for key, _ in items_positive] + [ key for key, _ in items_negative]
lp_weights       = [val for _, val in items_positive] + [-val for _, val in items_negative]

# L1正則化
total_weight     = np.sum(np.abs(lp_weights))
lp_weights       = np.array(lp_weights) / total_weight

# Embeddingモデルによる高次元空間への写像
model            = SentenceTransformer('BAAI/bge-m3')
lp_vector        = lp_weights.reshape(1, -1)                             # 1 × キーワード数M
lp_matrix        = model.encode(lp_keywords, normalize_embeddings=True)  # キーワード数M × 1024

# LPからみたスポットの重要度ベクトル
lp_coefficient   = lp_vector @ lp_matrix @ spots_matrix.T                # LPのスポット係数

# L2正則化
l2norm_vector    = np.linalg.norm(lp_coefficient, axis=1, keepdims=True)
lp_coefficient   = lp_coefficient / np.maximum(l2norm_vector, 1e-10)

# メモリ管理
del spots_matrix
del model, lp_matrix
del l2norm_vector
gc.collect()

lp_coefficient

### 📈 ADID 毎のスコア・理由を算出

コホート行列と LP 係数ベクトルの内積により、各 ADID の商品親和性スコアを計算する。

### 計算式

$$\text{score}[i] = \vec{cohort}_i \cdot \vec{lp\_coefficient}^T$$

各 ADID のコホートベクトル（L2 正規化済み）と LP のスポット係数ベクトルの **内積（コサイン類似度に相当）** を計算する。

### 処理フロー

```
np_cohort      : ADID数 × コホート数（疎行列・L2正規化済み）
lp_coefficient : 1 × スポット数（LP のスポット親和度ベクトル）

1. スコア算出
   np_scored = (np_cohort @ lp_coefficient.T).flatten()
   → 各 ADID に対して 1 つのスコアが得られる

2. 降順ソート
   → スコアが高い（＝商品と親和性が高い）ADID が先頭

3. 閾値設定
   REASON_THRESHOLD = 上位 20%（80 パーセンタイル）
   → 閾値以上の ADID のみを対象とする

4. Polars LazyFrame で adid × score × caption を結合

5. 地点（caption）ごとの訪問 ADID カウントを集計
```

### 出力変数

| 変数名 | 型 | 説明 |
|--------|----|------|
| `pldf_adidlist` | `polars.DataFrame` | `caption` × `adid` × `score`（閾値以上のみ） |
| `pldf_store_sum` | `polars.DataFrame` | `caption` × `total_count`（地点別集計） |
| `count` | `int` | 閾値以上のユニーク ADID 数 |

In [0]:
# ADID毎のスコアを算出
np_scored        = (np_cohort @ lp_coefficient.T).flatten()
indices          = np.argsort(np_scored)[::-1]
sorted_adidlist  = np_adidlist[indices]
sorted_scored    = np_scored[indices]

# メモリ管理
del np_cohort, lp_coefficient
del indices
del np_adidlist, np_scored
gc.collect()

# 閾値の設定(上位20%地点を指定する)
REASON_THRESHOLD = np.quantile(sorted_scored, 0.8)

# フレームワークへの変換
pldf_scores      = pl.LazyFrame({'adid': sorted_adidlist, 'score': sorted_scored})\
						.cast(  {'adid': pl.String,       'score': pl.Float32})\
                        .select(['adid', 'score'])
pldf_adidlist    = pl.from_pandas(pdf_adidlist).lazy()\
						.join(pldf_scores, on='adid', how='inner')\
                        .select(['caption', 'adid', 'score'])\
                        .filter(pl.col('score') > REASON_THRESHOLD)\
                        .collect()
# 解析対象地点毎のカウント数を計算
pldf_store_sum   = pldf_adidlist\
						.group_by(pl.col('caption'), maintain_order=True)\
                        .agg(pl.count().alias('total_count'))\
                        .select(['caption', 'total_count'])
count            = pldf_adidlist.select(pl.col("adid").n_unique()).item()

# メモリ管理
del pldf_scores
gc.collect()

print(f"処理対象        : {BRAND_NAME}")
print(f"ユニークなADID数 : {count}")
print(pldf_adidlist)

## 💾 保存処理

スコア分布の可視化と、結果データの Parquet 形式での保存を行う。

### 出力ファイル一覧

| ファイル名 | 形式 | 内容 |
|------------|------|------|
| `{BRAND_NAME}_visible.png` | PNG（300dpi） | スコア分布のヒストグラム（10% 刻み分位点付き） |
| `{BRAND_NAME}_all.parquet` | Parquet（Snappy 圧縮） | ADID × caption × score の全レコード |
| `{BRAND_NAME}_separate.parquet` | Parquet（Snappy 圧縮） | 地点（caption）別の訪問 ADID カウント集計 |

### 出力先ディレクトリ

```
/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/
├── {BRAND_NAME}_visible.png
├── {BRAND_NAME}_all.parquet
└── {BRAND_NAME}_separate.parquet
```

### ヒストグラムの見方

- **X 軸**: 各 ADID の商品親和性スコア
- **Y 軸**: 該当する ADID 数（頻度）
- **赤い点線**: 10% 刻みの分位点（10%〜90%）
- スコアが **右に偏る** ほど、商品と親和性の高い ADID が多いことを示す

In [ ]:
# 可視化
TARGET    = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_visible.png"
quantiles = np.quantile(sorted_scored, np.arange(0.1, 1.0, 0.1))
plt.figure(figsize=(10, 6))

plt.hist(sorted_scored, bins=30, color='skyblue', edgecolor='black')
for i, q in enumerate(quantiles):
    plt.axvline(q, color='red', linestyle='--', linewidth=1, alpha=0.6)
    # グラフ上部に「10%」「20%」などのラベルを表示（任意）
    plt.text(q, plt.ylim()[1] * 1.01, f'{(i+1)*10}%', 
             color='red', fontsize=9, ha='center', va='bottom')

plt.xlabel('Score')
plt.ylabel('Frequency')
plt.title('Histogram of Scored Data')
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig(TARGET, dpi=300, bbox_inches='tight')
plt.show()

TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_all.parquet"
pldf_adidlist.write_parquet( TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_separate.parquet"
pldf_store_sum.write_parquet(TARGET, compression="snappy")